3. Surround sound using HRTFs 
Build a tool that: 
-  Takes an HRTF configuration file (for a given subject), an audio sample, a starting azimuth 
angle and an azimuth rotation speed (angular velocity in degrees per second), basically 
simulating an sound source moving in space. 
-  Produces  a spatial  (surround  sound) effect  by  applying  different  HRTF  configurations 
across time frames. 
o  Note: we keep elevation fixed and focus on rotating azimuth. 
-  When  synthesizing  audio,  make  sure  to  use  overlap-add  (OLA)  windowing  or  similar 
strategies to ensure smooth transitions and avoid clipping artifacts. 
Refer  to  the  provided  example  file  for  guidance  on  achieving  continuous  and  realistic  spatial 
movement. 
Your tool can be a simple function with informative inputs or can be a simple graphical interface. 
It is up to you! 

In [53]:
import os

# user inputs for TAs :)
SUBJECT = "id17"
AUDIO = "music1.mp3"

SOFA_FILE = os.path.join("Data", "HRTFs", f"{SUBJECT}_measured.sofa")
AUDIO_FILE = os.path.join("Data", "Audio", AUDIO)
OUT_FILE = os.path.join("Data", "Audio", os.path.splitext(AUDIO)[0] + f"_{SUBJECT}_processed.wav")

START_AZ = 0 # in deg
SPEED = -90 # in deg
ELEVATION = 0 # any fixed elevation. if we don't have the exact el, I get the nearest one
FRAME_LEN = 2048 # frame size for STFT

In [6]:
import h5py
import numpy as np
import librosa
import soundfile as sf
from scipy.signal import fftconvolve, get_window

def hrtf_tool(sofa_file, audio_file, start_az, speed, elevation, frame_len=1024):
    f = h5py.File(sofa_file, "r")
    hrirs = f["Data.IR"][:]
    fs = int(f["Data.SamplingRate"][0])
    pos = f["SourcePosition"][:]
    az = pos[:, 0]
    el = pos[:, 1]

    el_idx = np.where(np.isclose(el, el[np.argmin(np.abs(el-elevation))]))[0]

    audio, _ = librosa.load(audio_file, sr=fs, mono=True)
    hop = frame_len//2
    window = get_window("hann", frame_len)
    audio = np.pad(audio, (0, frame_len))
    out = np.zeros((len(audio) + hrirs.shape[-1], 2))
    path = []

    for start in range(0, len(audio) - frame_len + 1, hop):
        t = (start + frame_len/2)/fs
        target = (start_az + speed*t) % 360
        idx = el_idx[np.argmin(np.abs(az[el_idx] - target))]
        path.append((t, az[idx]))

        frame = audio[start:start + frame_len]*window
        for ear in (0, 1):
            y = fftconvolve(frame, hrirs[idx, ear])
            out[start:start + len(y), ear] += y

    out /= np.max(np.abs(out))
    return out, fs, np.array(path)

In [54]:
from IPython.display import Audio

done, fs, path = hrtf_tool(SOFA_FILE, AUDIO_FILE, START_AZ, SPEED, ELEVATION, FRAME_LEN)
sf.write(OUT_FILE, done, fs)

Audio(done.T, rate=fs)